## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here

## B 长跑

In [ ]:
#include <bits/stdc++.h>
using namespace std;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int N, L, Maxn, S;

    while (cin >> N >> L >> Maxn >> S) {
        map<int, int> bestCost;

        for (int i = 0; i < N; i++) {
            int p, c;
            cin >> p >> c;

            if (p <= 0 || p >= L) continue;

            if (!bestCost.count(p)) {
                bestCost[p] = c;
            } else {
                bestCost[p] = min(bestCost[p], c);
            }
        }

        vector<int> pos;
        vector<int> cost;

        pos.push_back(0);
        cost.push_back(0);

        for (auto [p, c] : bestCost) {
            pos.push_back(p);
            cost.push_back(c);
        }

        pos.push_back(L);
        cost.push_back(0);

        int m = pos.size();
        const int INF = 1e9;

        vector<int> dp(m, INF);
        dp[0] = 0;

        for (int i = 1; i < m; i++) {
            for (int j = 0; j < i; j++) {
                if (dp[j] == INF) continue;

                if (pos[i] - pos[j] <= Maxn) {
                    dp[i] = min(dp[i], dp[j] + cost[i]);
                }
            }
        }

        cout << (dp[m - 1] <= S ? "Yes" : "No") << '\n';
    }

    return 0;
}

## C 最长回文

In [ ]:
#include <bits/stdc++.h>
using namespace std;

using ull = unsigned long long;

vector<int> manacherOdd(const string &s) {
    int n = s.size();
    vector<int> d(n);
    int l = 0, r = -1;

    for (int i = 0; i < n; i++) {
        int k = (i > r) ? 1 : min(d[l + r - i], r - i + 1);

        while (i - k >= 0 && i + k < n && s[i - k] == s[i + k]) {
            k++;
        }

        d[i] = k;

        if (i + k - 1 > r) {
            l = i - k + 1;
            r = i + k - 1;
        }
    }

    return d;
}

vector<int> manacherEven(const string &s) {
    int n = s.size();
    vector<int> d(n);
    int l = 0, r = -1;

    for (int i = 0; i < n; i++) {
        int k = (i > r) ? 0 : min(d[l + r - i + 1], r - i + 1);

        while (i - k - 1 >= 0 && i + k < n && s[i - k - 1] == s[i + k]) {
            k++;
        }

        d[i] = k;

        if (i + k - 1 > r) {
            l = i - k;
            r = i + k - 1;
        }
    }

    return d;
}


vector<int> getCenterLen(const string &s) {
    int n = s.size();
    vector<int> centerLen(2 * n + 1, 0);

    auto odd = manacherOdd(s);
    auto even = manacherEven(s);

    for (int i = 0; i < n; i++) {
        int sum = 2 * i;
        centerLen[sum + 1] = max(centerLen[sum + 1], 2 * odd[i] - 1);
    }

    for (int i = 0; i < n; i++) {
        int sum = 2 * i - 1;
        centerLen[sum + 1] = max(centerLen[sum + 1], 2 * even[i]);
    }

    return centerLen;
}

int getLen(const vector<int> &centerLen, int sum) {
    int idx = sum + 1;
    if (idx < 0 || idx >= (int)centerLen.size()) return 0;
    return centerLen[idx];
}

void orShiftedAndRange(
    vector<ull> &dst,
    const vector<ull> &bitsA,
    const vector<ull> &bitsR,
    int shift,
    int wl,
    int wr,
    int W
) {
    if (shift >= 0) {
        int wordShift = shift >> 6;
        int bitShift = shift & 63;

        for (int w = wl; w <= wr; w++) {
            int sw = w + wordShift;
            ull val = 0;

            if (sw < W) {
                val = bitsR[sw] >> bitShift;
                if (bitShift && sw + 1 < W) {
                    val |= bitsR[sw + 1] << (64 - bitShift);
                }
            }

            dst[w] |= bitsA[w] & val;
        }
    } else {
        int shiftAbs = -shift;
        int wordShift = shiftAbs >> 6;
        int bitShift = shiftAbs & 63;

        for (int w = wl; w <= wr; w++) {
            int sw = w - wordShift;
            ull val = 0;

            if (sw >= 0) {
                val = bitsR[sw] << bitShift;
                if (bitShift && sw - 1 >= 0) {
                    val |= bitsR[sw - 1] >> (64 - bitShift);
                }
            }

            dst[w] |= bitsA[w] & val;
        }
    }
}

int findFirstOne(const vector<ull> &bits, int l, int r) {
    if (l > r) return -1;

    int wl = l >> 6;
    int wr = r >> 6;

    for (int w = wl; w <= wr; w++) {
        ull val = bits[w];

        if (w == wl) {
            val &= (~0ULL << (l & 63));
        }

        if (w == wr) {
            int b = r & 63;
            ull mask = (b == 63) ? ~0ULL : ((1ULL << (b + 1)) - 1);
            val &= mask;
        }

        if (val) {
            return (w << 6) + __builtin_ctzll(val);
        }
    }

    return -1;
}

int findPrevZero(const vector<ull> &bits, int l, int r) {
    int wl = l >> 6;
    int wr = r >> 6;

    for (int w = wr; w >= wl; w--) {
        ull val = ~bits[w];

        if (w == wr) {
            int b = r & 63;
            ull mask = (b == 63) ? ~0ULL : ((1ULL << (b + 1)) - 1);
            val &= mask;
        }

        if (w == wl) {
            val &= (~0ULL << (l & 63));
        }

        if (val) {
            return (w << 6) + (63 - __builtin_clzll(val));
        }
    }

    return l - 1;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    string A, B;
    cin >> A >> B;

    if (n == 0) {
        cout << 0 << '\n';
        return 0;
    }

    string R = B;
    reverse(R.begin(), R.end());

    auto centerA = getCenterLen(A);
    auto centerB = getCenterLen(B);

    int ans = 0;

    for (int x : centerA) ans = max(ans, x);
    for (int x : centerB) ans = max(ans, x);

    int W = (n + 63) >> 6;

    vector<vector<ull>> bitsA(256, vector<ull>(W, 0));
    vector<vector<ull>> bitsR(256, vector<ull>(W, 0));

    vector<int> hasA(256, 0), hasR(256, 0);

    for (int i = 0; i < n; i++) {
        unsigned char ca = A[i];
        unsigned char cr = R[i];

        bitsA[ca][i >> 6] |= 1ULL << (i & 63);
        bitsR[cr][i >> 6] |= 1ULL << (i & 63);

        hasA[ca] = 1;
        hasR[cr] = 1;
    }

    vector<int> chars;
    for (int c = 0; c < 256; c++) {
        if (hasA[c] && hasR[c]) {
            chars.push_back(c);
        }
    }

    vector<ull> match(W, 0);


    for (int s = 0; s <= 2 * n - 2; s++) {
        int lo = max(0, s - (n - 1));
        int hi = min(n - 1, s / 2);

        if (lo > hi) continue;

        int midMax = max(getLen(centerB, s - 1), getLen(centerA, s + 1));

        int need;
        if (s <= midMax) {
            need = 0;
        } else {
            need = (s - midMax + 1) / 2;
        }

        int T = max(lo, need);

        if (T > hi) continue;

        int wl = lo >> 6;
        int wr = hi >> 6;

        for (int w = wl; w <= wr; w++) {
            match[w] = 0;
        }


        int shift = n - 1 - s;

        for (int ch : chars) {
            orShiftedAndRange(match, bitsA[ch], bitsR[ch], shift, wl, wr, W);
        }

        int first = findFirstOne(match, T, hi);

        if (first == -1) continue;

        int prevZero = findPrevZero(match, lo, first);
        int start = prevZero + 1;

        ans = max(ans, s - 2 * start + 2);
    }

    cout << ans << '\n';

    return 0;
}


## D 优惠券

In [ ]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <string>

using namespace std;


const int MAX_X = 1000005;
const int MAX_M = 500005;


int version[MAX_X];
int state_arr[MAX_X];
int last_op_pos[MAX_X];
int current_version = 0;


int dsu[MAX_M];

int find_set(int v) {
    if (v == dsu[v]) return v;
    return dsu[v] = find_set(dsu[v]);
}

void solve() {
    int m;
    while (cin >> m) {
        current_version++;
        vector<int> q_pos;
        dsu[0] = 0;
        
        bool error_found = false;
        int error_line = -1;
        
        for (int i = 1; i <= m; ++i) {
            char op[10];
            cin >> op;
            int x = -1;
            
            if (op[0] == 'I' || op[0] == 'O') {
                cin >> x;
            }
            
            if (error_found) continue;
            

            if (op[0] != 'I' && op[0] != 'O') {
                int k = q_pos.size();
                q_pos.push_back(i);
                dsu[k] = k;
                dsu[k + 1] = k + 1;
            } 
            else if (op[0] == 'I') {

                if (version[x] != current_version) {
                    version[x] = current_version;
                    state_arr[x] = 0;
                    last_op_pos[x] = 0;
                }
                
                if (state_arr[x] == 0) {
                    state_arr[x] = 1;
                    last_op_pos[x] = i;
                } else {

                    int start_idx = upper_bound(q_pos.begin(), q_pos.end(), last_op_pos[x]) - q_pos.begin();
                    

                    int avail_k = find_set(start_idx);
                    
                    if (avail_k < q_pos.size()) {

                        dsu[avail_k] = avail_k + 1;
                        last_op_pos[x] = i;
                    } else {

                        error_found = true;
                        error_line = i;
                    }
                }
            } 
            else if (op[0] == 'O') {
                if (version[x] != current_version) {
                    version[x] = current_version;
                    state_arr[x] = 0;
                    last_op_pos[x] = 0;
                }
                
                if (state_arr[x] == 1) {
                    state_arr[x] = 0;
                    last_op_pos[x] = i;
                } else {

                    int start_idx = upper_bound(q_pos.begin(), q_pos.end(), last_op_pos[x]) - q_pos.begin();
                    int avail_k = find_set(start_idx);
                    
                    if (avail_k < q_pos.size()) {

                        dsu[avail_k] = avail_k + 1;
                        last_op_pos[x] = i;
                    } else {
                        error_found = true;
                        error_line = i;
                    }
                }
            }
        }
        
        if (error_found) {
            cout << error_line << "\n";
        } else {
            cout << -1 << "\n";
        }
    }
}

int main() {

    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    
    solve();
    
    return 0;
}

## E 任意点

In [ ]:
#include <bits/stdc++.h>
using namespace std;

struct Point {
    int x, y;
};

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    vector<Point> p(n);

    for (int i = 0; i < n; i++) {
        cin >> p[i].x >> p[i].y;
    }

    vector<int> vis(n, 0);

    auto dfs = [&](auto &&self, int u) -> void {
        vis[u] = 1;

        for (int v = 0; v < n; v++) {
            if (vis[v]) continue;

            if (p[u].x == p[v].x || p[u].y == p[v].y) {
                self(self, v);
            }
        }
    };

    int components = 0;

    for (int i = 0; i < n; i++) {
        if (!vis[i]) {
            components++;
            dfs(dfs, i);
        }
    }

    cout << components - 1 << '\n';

    return 0;
}

## F 通配符匹配

In [ ]:
#include <bits/stdc++.h>
using namespace std;

struct Block {
    int offset;
    int id;
};

struct Segment {
    int len = 0;
    vector<Block> blocks;
    int anchor = -1;
};

vector<int> zFunction(const string& s) {
    int n = s.size();
    vector<int> z(n, 0);

    int l = 0, r = 0;

    for (int i = 1; i < n; i++) {
        if (i <= r) {
            z[i] = min(r - i + 1, z[i - l]);
        }

        while (i + z[i] < n && s[z[i]] == s[i + z[i]]) {
            z[i]++;
        }

        if (i + z[i] - 1 > r) {
            l = i;
            r = i + z[i] - 1;
        }
    }

    return z;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string pattern;
    cin >> pattern;

    int n;
    cin >> n;

    bool hasStar = false;
    for (char c : pattern) {
        if (c == '*') hasStar = true;
    }

    bool startStar = pattern.front() == '*';
    bool endStar = pattern.back() == '*';

    vector<string> rawParts;
    string cur;

    for (char c : pattern) {
        if (c == '*') {
            if (!cur.empty()) {
                rawParts.push_back(cur);
                cur.clear();
            }
        } else {
            cur.push_back(c);
        }
    }

    if (!cur.empty()) {
        rawParts.push_back(cur);
    }

    vector<Segment> segments;
    vector<string> literals;
    unordered_map<string, int> id;

    for (const string& part : rawParts) {
        Segment seg;
        seg.len = part.size();

        for (int i = 0; i < (int)part.size(); ) {
            if (part[i] == '?') {
                i++;
                continue;
            }

            int j = i;
            while (j < (int)part.size() && part[j] != '?') {
                j++;
            }

            string lit = part.substr(i, j - i);

            if (!id.count(lit)) {
                int newId = literals.size();
                id[lit] = newId;
                literals.push_back(lit);
            }

            seg.blocks.push_back({i, id[lit]});
            i = j;
        }

        int best = -1;
        int bestLen = -1;

        for (int i = 0; i < (int)seg.blocks.size(); i++) {
            int len = literals[seg.blocks[i].id].size();
            if (len > bestLen) {
                bestLen = len;
                best = i;
            }
        }

        seg.anchor = best;
        segments.push_back(seg);
    }

    while (n--) {
        string s;
        cin >> s;

        int m = s.size();
        int literalCount = literals.size();

        vector<vector<char>> occ(literalCount, vector<char>(m + 1, 0));
        vector<vector<int>> occPos(literalCount);

        for (int i = 0; i < literalCount; i++) {
            const string& lit = literals[i];
            int len = lit.size();

            if (len > m) continue;

            string t = lit + "#" + s;
            vector<int> z = zFunction(t);

            for (int pos = 0; pos + len <= m; pos++) {
                if (z[len + 1 + pos] >= len) {
                    occ[i][pos] = 1;
                    occPos[i].push_back(pos);
                }
            }
        }

        vector<vector<int>> valid(segments.size());

        for (int idx = 0; idx < (int)segments.size(); idx++) {
            const Segment& seg = segments[idx];

            if (seg.len > m) continue;

            if (seg.blocks.empty()) {
                continue;
            }

            const Block& anchor = seg.blocks[seg.anchor];

            for (int p : occPos[anchor.id]) {
                int start = p - anchor.offset;

                if (start < 0 || start + seg.len > m) {
                    continue;
                }

                bool ok = true;

                for (const Block& b : seg.blocks) {
                    int blockStart = start + b.offset;

                    if (blockStart < 0 || blockStart > m || !occ[b.id][blockStart]) {
                        ok = false;
                        break;
                    }
                }

                if (ok) {
                    valid[idx].push_back(start);
                }
            }
        }

        auto validAt = [&](int idx, int pos) -> bool {
            const Segment& seg = segments[idx];

            if (pos < 0 || pos + seg.len > m) {
                return false;
            }

            if (seg.blocks.empty()) {
                return true;
            }

            const vector<int>& v = valid[idx];
            return binary_search(v.begin(), v.end(), pos);
        };

        auto nextStart = [&](int idx, int pos) -> int {
            const Segment& seg = segments[idx];

            if (seg.len > m) {
                return -1;
            }

            if (seg.blocks.empty()) {
                if (pos + seg.len <= m) return pos;
                return -1;
            }

            const vector<int>& v = valid[idx];
            auto it = lower_bound(v.begin(), v.end(), pos);

            if (it == v.end()) return -1;
            return *it;
        };

        bool ok = true;

        if (!hasStar) {
            if (segments.empty()) {
                ok = (m == 0);
            } else {
                ok = (segments[0].len == m && validAt(0, 0));
            }

            cout << (ok ? "YES" : "NO") << '\n';
            continue;
        }

        if (segments.empty()) {
            cout << "YES\n";
            continue;
        }

        int left = 0;
        int right = (int)segments.size() - 1;
        int pos = 0;
        int limit = m;

        if (!startStar) {
            if (!validAt(0, 0)) {
                ok = false;
            } else {
                pos = segments[0].len;
                left = 1;
            }
        }

        if (ok && !endStar && left <= right) {
            int st = m - segments[right].len;

            if (!validAt(right, st)) {
                ok = false;
            } else {
                limit = st;
                right--;
            }
        }

        for (int i = left; ok && i <= right; i++) {
            int st = nextStart(i, pos);

            if (st == -1 || st + segments[i].len > limit) {
                ok = false;
                break;
            }

            pos = st + segments[i].len;
        }

        if (ok && pos > limit) {
            ok = false;
        }

        cout << (ok ? "YES" : "NO") << '\n';
    }

    return 0;
}

## G 汉诺塔

In [ ]:
#include <bits/stdc++.h>
using namespace std;

long long qpow(long long a, int b) {
    long long res = 1;

    while (b--) {
        res *= a;
    }

    return res;
}

int id(char c) {
    return c - 'A';
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    vector<vector<int>> rank(3, vector<int>(3, 100));

    for (int i = 0; i < 6; i++) {
        string op;
        cin >> op;

        int u = id(op[0]);
        int v = id(op[1]);

        rank[u][v] = i;
    }

    vector<int> nxt(3);

    for (int u = 0; u < 3; u++) {
        int best = -1;

        for (int v = 0; v < 3; v++) {
            if (u == v) continue;

            if (best == -1 || rank[u][v] < rank[u][best]) {
                best = v;
            }
        }

        nxt[u] = best;
    }

    int A = 0;
    long long ans;

    if (nxt[nxt[A]] == A) {
        ans = 2 * qpow(3, n - 1) - 1;
    } else if (nxt[nxt[nxt[A]]] == A) {
        ans = qpow(2, n) - 1;
    } else {
        ans = qpow(3, n - 1);
    }

    cout << ans << '\n';

    return 0;
}

## H 马步距离

In [ ]:
#include <bits/stdc++.h>
using namespace std;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    long long xp, yp, xs, ys;
    cin >> xp >> yp >> xs >> ys;

    long long dx = llabs(xp - xs);
    long long dy = llabs(yp - ys);

    long long x = max(dx, dy);
    long long y = min(dx, dy);

    if (x == 0 && y == 0) {
        cout << 0 << '\n';
        return 0;
    }

    if (x == 1 && y == 0) {
        cout << 3 << '\n';
        return 0;
    }

    if (x == 2 && y == 2) {
        cout << 4 << '\n';
        return 0;
    }

    long long ans = max((x + 1) / 2, (x + y + 2) / 3);

    if ((ans + x + y) % 2 == 1) {
        ans++;
    }

    cout << ans << '\n';

    return 0;
}

## I 直方图最大矩形

In [ ]:
class Solution {
public:
    int largestRectangleArea(vector<int>& heights) {
        int n = heights.size();

        if (n == 0) {
            return 0;
        }

        stack<int> st;
        int ans = 0;

        for (int i = 0; i <= n; i++) {
            int cur = (i == n ? 0 : heights[i]);

            while (!st.empty() && heights[st.top()] > cur) {
                int h = heights[st.top()];
                st.pop();

                int width;

                if (st.empty()) {
                    width = i;
                } else {
                    width = i - st.top() - 1;
                }

                ans = max(ans, h * width);
            }

            st.push(i);
        }

        return ans;
    }
};

## J 消防局的设立

In [ ]:
#include <bits/stdc++.h>
using namespace std;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    vector<vector<int>> g(n + 1);

    for (int i = 2; i <= n; i++) {
        int a;
        cin >> a;
        g[i].push_back(a);
        g[a].push_back(i);
    }

    vector<int> parent(n + 1, 0), depth(n + 1, 0), order;

    queue<int> q;
    q.push(1);
    parent[1] = 0;

    while (!q.empty()) {
        int u = q.front();
        q.pop();

        order.push_back(u);

        for (int v : g[u]) {
            if (v == parent[u]) continue;

            parent[v] = u;
            depth[v] = depth[u] + 1;
            q.push(v);
        }
    }

    vector<char> covered(n + 1, 0);

    vector<char> expanded(n + 1, 0);

    int ans = 0;

    auto addStation = [&](int x) {
        ans++;

        covered[x] = 1;

        for (int y : g[x]) {
            covered[y] = 1;

            if (!expanded[y]) {
                expanded[y] = 1;

                for (int z : g[y]) {
                    covered[z] = 1;
                }
            }
        }
    };

    for (int i = (int)order.size() - 1; i >= 0; i--) {
        int u = order[i];

        if (covered[u]) continue;

        int station = u;

        if (parent[station] != 0) {
            station = parent[station];
        }

        if (parent[station] != 0) {
            station = parent[station];
        }

        addStation(station);
    }

    cout << ans << '\n';

    return 0;
}